# 2. Modelado Supervisado y Aprendizaje por Refuerzo
En este cuaderno implementamos los modelos clásicos (RF, XGB, SVM, LR, KNN) y simulamos ambientes con Agentes de RL (PPO, DQN). Para no repetir el cruce, importamos los datos preprocesados de Kedro.

In [10]:
import os
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
import warnings
warnings.filterwarnings("ignore")

In [12]:
data = pd.read_pickle("data/02_intermediate/preprocessed_data.pkl")
X = data.drop(["target", "id_estudiante", "nombre", "rut", "email", "estado_matricula"], axis=1, errors="ignore")
y_cls = data["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y_cls, test_size=0.2, random_state=42, stratify=y_cls)

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
num_f = X.select_dtypes(include=["number"]).columns
cat_f = X.select_dtypes(include=["object"]).columns
preprocessor = ColumnTransformer([("num", Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler())]), num_f), 
                                  ("cat", Pipeline([("i", SimpleImputer(strategy="constant")), ("o", OneHotEncoder(handle_unknown="ignore"))]), cat_f)])

### Modelos Supervisados

In [13]:
models = {
    "RandomForest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42),
    "SVM": SVC(random_state=42),
    "LogisticRegression": LogisticRegression(random_state=42, max_iter=1000),
    "KNN": KNeighborsClassifier()
}

for name, model in models.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("classifier", model)])
    pipe.fit(X_train, y_train)
    print(f"--- {name} ---")
    print(classification_report(y_test, pipe.predict(X_test)))

--- RandomForest ---
              precision    recall  f1-score   support

           0       0.91      0.98      0.95        63
           1       0.50      0.14      0.22         7

    accuracy                           0.90        70
   macro avg       0.71      0.56      0.58        70
weighted avg       0.87      0.90      0.87        70

--- XGBoost ---
              precision    recall  f1-score   support

           0       0.91      0.97      0.94        63
           1       0.33      0.14      0.20         7

    accuracy                           0.89        70
   macro avg       0.62      0.56      0.57        70
weighted avg       0.85      0.89      0.86        70

--- SVM ---
              precision    recall  f1-score   support

           0       0.90      1.00      0.95        63
           1       0.00      0.00      0.00         7

    accuracy                           0.90        70
   macro avg       0.45      0.50      0.47        70
weighted avg       0.81  

### Aprendizaje por Refuerzo (RL)

In [14]:
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO, DQN

class ClassificationEnv(gym.Env):
    def __init__(self, X, y):
        super(ClassificationEnv, self).__init__()
        self.X = X
        self.y = y.values
        self.current_step = 0
        self.action_space = spaces.Discrete(2)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(X.shape[1],), dtype=np.float32)
    
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        return self.X[self.current_step], {}
    
    def step(self, action):
        correct = (action == self.y[self.current_step])
        self.current_step += 1
        done = self.current_step >= len(self.X)
        return self.X[self.current_step] if not done else np.zeros(self.X.shape[1], dtype=np.float32), 1.0 if correct else -1.0, done, False, {}

X_prep_train = preprocessor.fit_transform(X_train).toarray()
env = ClassificationEnv(X_prep_train[:100], y_train[:100])

ppo = PPO("MlpPolicy", env, verbose=0).learn(total_timesteps=500)
print("Agente PPO entrenado con éxito.")

Agente PPO entrenado con éxito.
